# 🧹 Fonctions SQL Indispensables — Nettoyage de Base de Données
Référence complète des fonctions SQL pour auditer, nettoyer et valider vos données.

---
## 1. 🔤 Nettoyage des chaînes de caractères

In [ ]:
-- TRIM / LTRIM / RTRIM : supprimer les espaces
SELECT TRIM('  hello  ');          -- 'hello'
SELECT LTRIM('  hello');           -- 'hello'
SELECT RTRIM('hello  ');           -- 'hello'

-- REPLACE : remplacer des caractères indésirables
SELECT REPLACE(telephone, ' ', '') FROM clients;
SELECT REPLACE(telephone, '-', '') FROM clients;

-- UPPER / LOWER / INITCAP : normaliser la casse
SELECT LOWER(email) FROM clients;          -- tout en minuscules
SELECT UPPER(code_pays) FROM clients;      -- tout en majuscules
SELECT INITCAP(nom) FROM clients;          -- Jean Dupont (PG/Oracle)

-- REGEXP_REPLACE : nettoyage avancé via regex
SELECT REGEXP_REPLACE(telephone, '[^0-9]', '', 'g') FROM clients;  -- garde uniquement les chiffres

-- LENGTH : détecter des valeurs anormales
SELECT * FROM clients WHERE LENGTH(TRIM(code_postal)) <> 5;

-- SUBSTRING / LEFT / RIGHT : extraire des portions
SELECT LEFT(code_postal, 2) AS departement FROM clients;
SELECT SUBSTRING(iban, 1, 4) AS pays_banque FROM comptes;

-- CONCAT : recoller des champs fragmentés
SELECT CONCAT(TRIM(prenom), ' ', TRIM(nom)) AS nom_complet FROM clients;

---
## 2. ❓ Gestion des valeurs NULL et manquantes

In [ ]:
-- COALESCE : retourne le premier non-NULL
SELECT COALESCE(email, 'email_inconnu@na.com') FROM clients;
SELECT COALESCE(telephone_mobile, telephone_fixe, 'Non renseigné') FROM clients;

-- NULLIF : convertit une valeur vide en NULL
SELECT NULLIF(TRIM(commentaire), '') FROM commandes;  -- '' devient NULL

-- IFNULL (MySQL) / NVL (Oracle)
SELECT IFNULL(ville, 'Ville inconnue') FROM clients;  -- MySQL
SELECT NVL(salaire, 0) FROM employes;                 -- Oracle

-- Compter les NULL par colonne (audit)
SELECT
  COUNT(*) AS total,
  COUNT(*) - COUNT(email)     AS null_email,
  COUNT(*) - COUNT(telephone) AS null_telephone,
  COUNT(*) - COUNT(ville)     AS null_ville
FROM clients;

-- Supprimer les lignes entièrement vides
DELETE FROM clients
WHERE nom IS NULL AND prenom IS NULL AND email IS NULL;

---
## 3. 🔄 Conversion de types

In [ ]:
-- CAST : conversion standard
SELECT CAST('42' AS INT);
SELECT CAST('2024-01-15' AS DATE);
SELECT CAST(prix AS DECIMAL(10,2)) FROM produits;

-- TRY_CAST (SQL Server) : conversion sans erreur si échoue
SELECT TRY_CAST(valeur_brute AS INT) FROM import_brut;  -- retourne NULL si invalide

-- CONVERT (SQL Server / MySQL)
SELECT CONVERT(INT, '42');                          -- SQL Server
SELECT CONVERT('2024-01-15', DATE);                 -- MySQL

-- TO_DATE / TO_NUMBER (PostgreSQL / Oracle)
SELECT TO_DATE('15/01/2024', 'DD/MM/YYYY');          -- PostgreSQL/Oracle
SELECT TO_NUMBER('1 234,50', '9G999D99');            -- Oracle

-- STR_TO_DATE (MySQL)
SELECT STR_TO_DATE('15-01-2024', '%d-%m-%Y') FROM commandes;

---
## 4. 🗑️ Détection et suppression des doublons

In [ ]:
-- Détecter les doublons
SELECT email, COUNT(*) AS nb
FROM clients
GROUP BY email
HAVING COUNT(*) > 1;

-- Voir tous les enregistrements dupliqués
SELECT * FROM clients
WHERE email IN (
  SELECT email FROM clients
  GROUP BY email HAVING COUNT(*) > 1
);

-- Garder uniquement le plus récent (MySQL / SQL Server)
DELETE FROM clients
WHERE id NOT IN (
  SELECT MAX(id)
  FROM clients
  GROUP BY email
);

-- Avec ROW_NUMBER (PostgreSQL / SQL Server)
WITH dedup AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY email ORDER BY id DESC) AS rn
  FROM clients
)
DELETE FROM dedup WHERE rn > 1;

-- Créer une table propre sans doublons
CREATE TABLE clients_clean AS
SELECT DISTINCT * FROM clients;

---
## 5. 📅 Nettoyage et validation des dates

In [ ]:
-- Détecter les dates invalides (SQL Server)
SELECT * FROM commandes WHERE ISDATE(date_commande) = 0;

-- Détecter les dates aberrantes (dans le futur ou trop anciennes)
SELECT * FROM clients
WHERE date_naissance > CURRENT_DATE
   OR date_naissance < '1900-01-01';

-- Détecter les incohérences (date fin < date début)
SELECT * FROM contrats
WHERE date_fin < date_debut;

-- Extraire des composantes de date
SELECT
  EXTRACT(YEAR  FROM date_naissance) AS annee,
  EXTRACT(MONTH FROM date_naissance) AS mois,
  EXTRACT(DAY   FROM date_naissance) AS jour
FROM clients;

-- Calculer un écart (vérifier cohérence)
SELECT DATEDIFF(DAY, date_debut, date_fin) AS duree_jours FROM contrats;  -- SQL Server/MySQL
SELECT (date_fin - date_debut) AS duree_jours FROM contrats;              -- PostgreSQL

-- Normaliser une date stockée en chaîne
UPDATE commandes
SET date_commande = TO_DATE(date_brute, 'DD/MM/YYYY')
WHERE date_brute IS NOT NULL;

---
## 6. ✅ Validation de formats

In [ ]:
-- Valider les emails
SELECT * FROM clients
WHERE email NOT LIKE '%@%.%'
   OR email LIKE '% %';  -- pas d'espace dans un email

-- Valider les codes postaux français (5 chiffres)
SELECT * FROM clients
WHERE code_postal NOT REGEXP '^[0-9]{5}$';  -- MySQL

SELECT * FROM clients
WHERE code_postal !~ '^[0-9]{5}$';           -- PostgreSQL

-- Valider les numéros de téléphone français
SELECT * FROM clients
WHERE REGEXP_REPLACE(telephone, '[^0-9]', '') NOT REGEXP '^(0[1-9][0-9]{8})$';

-- Valider les valeurs numériques hors plage
SELECT * FROM produits WHERE prix < 0 OR prix > 100000;
SELECT * FROM employes WHERE age NOT BETWEEN 16 AND 70;

-- Vérifier l'intégrité référentielle (orphelins)
SELECT c.* FROM commandes c
LEFT JOIN clients cl ON c.client_id = cl.id
WHERE cl.id IS NULL;  -- commandes sans client correspondant

---
## 7. 📊 Audit global de qualité des données

In [ ]:
-- Taux de remplissage par colonne
SELECT
  COUNT(*) AS total_lignes,
  ROUND(100.0 * COUNT(nom)       / COUNT(*), 1) AS pct_nom,
  ROUND(100.0 * COUNT(email)     / COUNT(*), 1) AS pct_email,
  ROUND(100.0 * COUNT(telephone) / COUNT(*), 1) AS pct_telephone,
  ROUND(100.0 * COUNT(ville)     / COUNT(*), 1) AS pct_ville
FROM clients;

-- Résumé statistique d'une colonne numérique
SELECT
  MIN(prix)   AS min_prix,
  MAX(prix)   AS max_prix,
  AVG(prix)   AS moy_prix,
  STDDEV(prix) AS ecart_type
FROM produits;

-- Valeurs distinctes (détecter les incohérences de référentiel)
SELECT pays, COUNT(*) AS nb
FROM clients
GROUP BY pays
ORDER BY nb DESC;

-- Détecter les valeurs suspectes (espaces cachés, caractères spéciaux)
SELECT * FROM clients
WHERE nom <> TRIM(nom)         -- espaces en trop
   OR nom LIKE '%  %';          -- doubles espaces internes

---
## 8. 🚀 Script complet de nettoyage (exemple)

In [ ]:
-- Étape 1 : Créer une table de travail
CREATE TABLE clients_clean AS SELECT * FROM clients;

-- Étape 2 : Nettoyer les chaînes
UPDATE clients_clean SET
  nom       = INITCAP(TRIM(nom)),
  prenom    = INITCAP(TRIM(prenom)),
  email     = LOWER(TRIM(email)),
  telephone = REGEXP_REPLACE(TRIM(telephone), '[^0-9]', '', 'g'),
  ville     = INITCAP(TRIM(ville));

-- Étape 3 : Remplacer les chaînes vides par NULL
UPDATE clients_clean SET
  email     = NULLIF(email, ''),
  telephone = NULLIF(telephone, ''),
  ville     = NULLIF(ville, '');

-- Étape 4 : Supprimer les doublons (garder le plus récent)
DELETE FROM clients_clean
WHERE id NOT IN (
  SELECT MAX(id) FROM clients_clean GROUP BY email
);

-- Étape 5 : Supprimer les lignes sans identifiant métier
DELETE FROM clients_clean
WHERE email IS NULL AND telephone IS NULL;

-- Étape 6 : Vérification finale
SELECT
  COUNT(*) AS total,
  COUNT(email) AS avec_email,
  COUNT(telephone) AS avec_telephone
FROM clients_clean;

---
## 💡 Ordre recommandé

| Étape | Action |
|-------|--------|
| 1 | **Audit** → nulls, doublons, formats |
| 2 | **Typage** → corriger les types |
| 3 | **Chaînes** → trim, casse, regex |
| 4 | **Nulls** → remplacer ou supprimer |
| 5 | **Doublons** → dédupliquer |
| 6 | **Cohérence** → dates, plages, références |